# CardioIA — IA em Séries Temporais: Regressão Logística vs. LIF

**Projeto:** CardioIA — FIAP Fase 3 · Ir Além 2  
**Autores:** João (RM565999) · Tayná (RM562491) · Carlos (RM566487) · Endrew (RM563646)  

## Objetivo
Comparar dois classificadores para detecção de **taquicardia** em séries temporais de BPM:
- **Regressão Logística** — modelo estatístico tradicional, altamente interpretável
- **LIF (Leaky Integrate-and-Fire)** — rede neuromórfica bio-inspirada, implementada com NumPy puro

## Contexto CardioIA
O sistema CardioIA monitora BPM, temperatura e umidade via ESP32, transmitindo dados por MQTT (HiveMQ Cloud) ao Node-RED. Este notebook analisa séries temporais desses sinais para avaliar modelos de IA na detecção automática de taquicardia (BPM > 120).

In [ ]:
# Célula 2 — Imports (disponíveis no Colab sem instalação extra)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix,
                              ConfusionMatrixDisplay)
import time
import os

# Criar pastas de saída
os.makedirs('docs/images', exist_ok=True)

# Paleta médica: azul=#00d4ff (normal), vermelho=#ff4444 (taquicardia)
PALETA = {0: '#00d4ff', 1: '#ff4444'}
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Imports OK')

In [ ]:
# Célula 3 — Geração de dados simulados (seed=42, reprodutível)
NP_SEED = 42
np.random.seed(NP_SEED)
N = 500

# Labels: 80% normal (0), 20% taquicardia (1)
y = np.random.choice([0, 1], size=N, p=[0.80, 0.20])

# BPM base por classe + ruído gaussiano (simula sensor real)
bpm_base = np.where(y==0, np.random.normal(82, 14, N), np.random.normal(138, 10, N))
bpm_atual  = bpm_base + np.random.normal(0, 2.5, N)      # ruído sensor
bpm_media  = bpm_base + np.random.normal(0, 4.0, N)      # média 5s
bpm_std    = np.where(y==0, np.abs(np.random.normal(3, 1, N)),
                             np.abs(np.random.normal(8, 2, N)))
temperatura = np.where(y==0, np.random.normal(36.5, 0.4, N),
                              np.random.normal(37.9, 0.7, N))
umidade = np.random.normal(58, 12, N)

df = pd.DataFrame({
    'bpm_atual': bpm_atual, 'bpm_media_5s': bpm_media,
    'bpm_std_5s': bpm_std, 'temperatura': temperatura,
    'umidade': umidade, 'label': y
})

print('Shape:', df.shape)
print('\nDistribuição de classes:')
print(df['label'].value_counts())
print('\nEstatísticas:')
df.describe().round(2)

In [ ]:
# Célula 4 — Análise Exploratória de Dados (EDA)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CardioIA — Análise Exploratória dos Dados Simulados', fontsize=14, fontweight='bold')

# 1. Histograma BPM com threshold
ax = axes[0,0]
for cls, cor in PALETA.items():
    ax.hist(df[df['label']==cls]['bpm_atual'], bins=30, alpha=0.7,
            color=cor, label=['Normal','Taquicardia'][cls])
ax.axvline(120, color='black', linestyle='--', linewidth=1.5, label='Limiar 120 BPM')
ax.set_title('Distribuição de BPM por Classe')
ax.set_xlabel('BPM Atual')
ax.set_ylabel('Frequência')
ax.legend()

# 2. Boxplot BPM por classe
ax = axes[0,1]
df.boxplot(column='bpm_atual', by='label', ax=ax,
           boxprops=dict(color='#334155'),
           medianprops=dict(color='#ef4444', linewidth=2))
ax.set_title('Boxplot BPM × Classe')
ax.set_xlabel('Classe (0=Normal, 1=Taquicardia)')
ax.set_ylabel('BPM')
plt.sca(ax)
plt.title('Boxplot BPM × Classe')

# 3. Distribuição das classes
ax = axes[1,0]
counts = df['label'].value_counts()
ax.pie(counts, labels=['Normal','Taquicardia'], colors=[PALETA[0], PALETA[1]],
       autopct='%1.1f%%', startangle=90, textprops={'fontsize':11})
ax.set_title('Proporção das Classes')

# 4. Matriz de correlação
ax = axes[1,1]
corr = df.drop('label', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlação')

plt.tight_layout()
plt.show()
print('EDA concluída.')

In [ ]:
# Célula 5 — Pré-processamento
X = df.drop('label', axis=1).values
y = df['label'].values

# Split 80/20 estratificado (mesma proporção de classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Padronização (StandardScaler)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Treino: {X_train_s.shape} | Teste: {X_test_s.shape}')
print(f'Classes no teste — Normal: {(y_test==0).sum()} | Taquicardia: {(y_test==1).sum()}')

In [ ]:
# Célula 6 — MODELO 1: Regressão Logística
t0 = time.time()
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
tempo_lr = (time.time() - t0) * 1000

lr_pred  = lr.predict(X_test_s)
lr_prob  = lr.predict_proba(X_test_s)[:,1]

lr_acc  = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred, zero_division=0)
lr_rec  = recall_score(y_test, lr_pred, zero_division=0)
lr_f1   = f1_score(y_test, lr_pred, zero_division=0)
lr_auc  = roc_auc_score(y_test, lr_prob)

print('=== Regressão Logística ===')
print(f'Accuracy:  {lr_acc:.4f}  |  Precision: {lr_prec:.4f}')
print(f'Recall:    {lr_rec:.4f}  |  F1-Score:  {lr_f1:.4f}')
print(f'ROC-AUC:   {lr_auc:.4f}  |  Tempo:     {tempo_lr:.1f} ms')

print('\nCoeficientes (interpretabilidade médica):')
features = ['bpm_atual','bpm_media_5s','bpm_std_5s','temperatura','umidade']
for f, c in sorted(zip(features, lr.coef_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {f:18s}: {c:+.4f}')

## Interpretabilidade Médica — Regressão Logística

Os **coeficientes** da Regressão Logística têm significado clínico direto:
- Coeficientes positivos elevados (ex.: `bpm_atual`, `bpm_media_5s`) indicam que valores altos desses sinais aumentam a probabilidade de taquicardia.
- Um médico cardiologista pode auditar o modelo simplesmente lendo os pesos — característica impossível no modelo LIF.
- Esta interpretabilidade é exigida pela LGPD e pelas boas práticas de IA em saúde.

In [ ]:
# Célula 7 — MODELO 2: Rede Neuromórfica LIF (NumPy puro)
# Equação: τ·dV/dt = -(V - V_rest) + R·I(t)

class LIFNeuron:
    """Neurônio Leaky Integrate-and-Fire — implementação NumPy pura."""
    def __init__(self, tau=20.0, V_rest=-70.0, V_thresh=-55.0, V_reset=-75.0, R=10.0, dt=1.0):
        self.tau, self.V_rest, self.V_thresh = tau, V_rest, V_thresh
        self.V_reset, self.R, self.dt = V_reset, R, dt
        self.V = V_rest

    def reset(self):
        self.V = self.V_rest

    def step(self, I):
        """Um passo temporal dt=1ms. Retorna 1 se houve spike."""
        dV = (-(self.V - self.V_rest) + self.R * I) * (self.dt / self.tau)
        self.V += dV
        if self.V >= self.V_thresh:
            self.V = self.V_reset
            return 1
        return 0

def classify_lif(X_scaled, weights, bias, T=100, fire_threshold=0.05):
    """Classifica amostras via firing rate do neurônio LIF."""
    probs, preds = [], []
    neuron = LIFNeuron()
    for sample in X_scaled:
        # Converter features ponderados em corrente de entrada
        I_input = max(0.0, float(np.dot(sample, weights) + bias) * 8.0)
        neuron.reset()
        spikes = sum(neuron.step(I_input) for _ in range(T))
        rate = spikes / T
        probs.append(rate)
        preds.append(1 if rate > fire_threshold else 0)
    return np.array(preds), np.array(probs)

# Usar pesos da LR como sinapses (paradigma híbrido)
t0 = time.time()
lif_pred, lif_prob_raw = classify_lif(X_test_s, lr.coef_[0], lr.intercept_[0])
tempo_lif = (time.time() - t0) * 1000

# Normalizar probabilidades para ROC
lif_prob = lif_prob_raw / (lif_prob_raw.max() + 1e-9)

lif_acc  = accuracy_score(y_test, lif_pred)
lif_prec = precision_score(y_test, lif_pred, zero_division=0)
lif_rec  = recall_score(y_test, lif_pred, zero_division=0)
lif_f1   = f1_score(y_test, lif_pred, zero_division=0)
lif_auc  = roc_auc_score(y_test, lif_prob)

print('=== LIF Neuromórfico ===')
print(f'Accuracy:  {lif_acc:.4f}  |  Precision: {lif_prec:.4f}')
print(f'Recall:    {lif_rec:.4f}  |  F1-Score:  {lif_f1:.4f}')
print(f'ROC-AUC:   {lif_auc:.4f}  |  Tempo:     {tempo_lif:.1f} ms')

In [ ]:
# Célula 8 — Comparação lado a lado

# ── Tabela comparativa ─────────────────────────────────────────────────────
result = pd.DataFrame({
    'Modelo':        ['Regressão Logística', 'LIF Neuromórfico'],
    'Accuracy':      [lr_acc, lif_acc],
    'Precision':     [lr_prec, lif_prec],
    'Recall':        [lr_rec, lif_rec],
    'F1':            [lr_f1, lif_f1],
    'AUC':           [lr_auc, lif_auc],
    'Tempo_treino_ms': [round(tempo_lr,1), round(tempo_lif,1)]
})
display(result.set_index('Modelo').style.format('{:.4f}', subset=['Accuracy','Precision','Recall','F1','AUC']))

# ── Gráfico de barras agrupadas ─────────────────────────────────────────────
metricas = ['Accuracy','Precision','Recall','F1','AUC']
x = np.arange(len(metricas))
w = 0.35
fig, ax = plt.subplots(figsize=(11,5))
bars1 = ax.bar(x - w/2, result[metricas].iloc[0], w, label='Regressão Logística',
               color='#00d4ff', edgecolor='white')
bars2 = ax.bar(x + w/2, result[metricas].iloc[1], w, label='LIF Neuromórfico',
               color='#ff4444', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(metricas)
ax.set_ylim(0, 1.15)
ax.set_title('Comparação de Métricas — Regressão Logística vs. LIF', fontsize=13, fontweight='bold')
ax.set_ylabel('Valor da Métrica')
ax.legend(loc='lower right')
for bar in list(bars1)+list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('docs/images/ia_comparacao_modelos.png', dpi=150)
plt.show()

# ── Curvas ROC sobrepostas ──────────────────────────────────────────────────
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_prob)
fpr_lif,tpr_lif,_ = roc_curve(y_test, lif_prob)
fig, ax = plt.subplots(figsize=(7,6))
ax.plot(fpr_lr,  tpr_lr,  color='#00d4ff', lw=2.5, label=f'Reg. Logística (AUC={lr_auc:.3f})')
ax.plot(fpr_lif, tpr_lif, color='#ff4444', lw=2.5, label=f'LIF Neuromórfico (AUC={lif_auc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Aleatório')
ax.set_xlabel('Taxa de Falso Positivo'); ax.set_ylabel('Taxa de Verdadeiro Positivo')
ax.set_title('Curvas ROC — Comparação dos Modelos', fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout()
plt.savefig('docs/images/ia_curvas_roc.png', dpi=150); plt.show()

# ── Matrizes de Confusão ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12,5))
fig.suptitle('Matrizes de Confusão', fontsize=13, fontweight='bold')
for i, (pred, nome, cor) in enumerate([(lr_pred,'Regressão Logística','Blues'),
                                        (lif_pred,'LIF Neuromórfico','Reds')]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cor, ax=axes[i],
                xticklabels=['Normal','Taquicardia'],
                yticklabels=['Normal','Taquicardia'])
    axes[i].set_title(nome); axes[i].set_xlabel('Previsto'); axes[i].set_ylabel('Real')
plt.tight_layout()
plt.savefig('docs/images/ia_matriz_confusao.png', dpi=150); plt.show()
print('Figuras salvas em docs/images/')

## Análise Crítica

### 1. Qual modelo apresentou melhor desempenho numérico?
A **Regressão Logística** apresentou métricas superiores em todas as dimensões avaliadas (Accuracy, Precision, Recall, F1 e AUC). Isso decorre da natureza essencialmente linear da fronteira de decisão nos dados gerados: taquicardia é definida por BPM > 120, uma regra linear que a Regressão Logística modela com eficiência máxima.

### 2. Qual modelo é mais explicável para um médico cardiologista?
A **Regressão Logística** é o padrão-ouro em interpretabilidade clínica. Seus coeficientes quantificam a contribuição de cada sinal vital na decisão de classificação, permitindo ao médico compreender e auditar o raciocínio do modelo. O LIF opera como uma *caixa cinza* — o disparo de spikes temporais não possui equivalência direta com conceitos clínicos familiares.

### 3. Trade-offs: precisão × transparência × custo computacional
| Dimensão | Regressão Logística | LIF Neuromórfico |
|:---|:---:|:---:|
| **Precisão** | Alta | Moderada |
| **Interpretabilidade** | Total (coeficientes) | Baixa (taxa de disparo) |
| **Custo computacional** | Muito baixo — O(n·features) | Alto — O(n·T·features) |
| **Treinamento** | Milissegundos | Segundos a minutos |

### 4. Viabilidade de embarcamento no ESP32
- **Regressão Logística:** Reduz-se a um produto escalar (5 multiplicações + 1 comparação). Totalmente viável no ESP32 — inferência em microsegundos com consumo de RAM inferior a 100 bytes.
- **LIF:** Exige simulação de integração temporal em 100 passos por amostra. Pesado para MCUs sem FPU dedicada; viável apenas com simplificações radicais ou aceleradores neuromórficos dedicados (Intel Loihi, IBM TrueNorth).

### 5. Qual modelo é recomendado para uso clínico real?
**Regressão Logística**, pelas seguintes razões:
- ✅ Auditável: coeficientes explicam cada decisão ao médico
- ✅ Embarcável: inferência direta no ESP32 sem hardware especial
- ✅ Validado: amplamente utilizado em medicina baseada em evidências
- ✅ Regulatório: aderente às exigências de explicabilidade da LGPD e normas de IA em saúde

O modelo LIF é valioso como objeto de pesquisa e para estudo de eficiência energética em chips neuromórficos, mas ainda imaturo para certificação em dispositivos médicos de uso clínico.

## Conclusão

Este notebook demonstrou a viabilidade de detecção automática de taquicardia em séries temporais de sinais vitais utilizando duas abordagens contrastantes de classificação. A **Regressão Logística** mostrou-se superior em desempenho, interpretabilidade e viabilidade de embarcamento — qualidades indispensáveis em dispositivos médicos críticos.

O modelo **LIF Neuromórfico**, implementado puramente com NumPy, demonstrou a factibilidade conceitual de redes bio-inspiradas para processamento de séries temporais biomédicas, abrindo caminho para pesquisas futuras com hardware neuromórfico dedicado.

### Próximos Passos
- Coletar dados reais do CardioIA via HiveMQ Cloud e retreinar os modelos
- Avaliar modelos de séries temporais (LSTM, Transformer) para padrões temporais mais complexos
- Exportar coeficientes da Regressão Logística para inferência direta no ESP32 (TensorFlow Lite for Microcontrollers)

### Referências
- Scikit-learn Documentation — https://scikit-learn.org
- Mainen, Z.F.; Sejnowski, T.J. *Reliability of spike timing in neocortical neurons.* Science, 1995.
- Gerstner, W.; Kistler, W. *Spiking Neuron Models.* Cambridge University Press, 2002.
- Documentação CardioIA — https://github.com/JV-004/FIAP-CardioIA-Fase3